In [1]:
import pandas as pd
url = "https://edvyro.in/datasets/healthcare_patients_sample.csv"
df = pd.read_csv(url)
df.head()

HTTPError: HTTP Error 403: Forbidden

In [2]:
import pandas as pd
from io import StringIO

data = """PatientID,Age,Gender,Diagnosis,PrimaryMedication,DosageMg,TreatmentDurationDays,Readmitted,BloodPressure,CholesterolLevel
PAT-201,42,Male,Hypertension,Amlodipine,10,90,No,138/88,210
PAT-202,58,Female,Type 2 Diabetes,Metformin,500,180,No,128/80,195
PAT-203,67,Male,Heart Disease,Atorvastatin,40,120,Yes,145/92,245
PAT-204,35,Female,Asthma,Albuterol,2,30,No,118/75,180
PAT-205,71,Female,Osteoarthritis,Ibuprofen,400,60,Yes,132/84,205
PAT-206,50,Male,Type 2 Diabetes,Metformin,850,90,No,125/82,215
PAT-207,63,Female,Hypertension,Lisinopril,20,150,No,135/86,220
PAT-208,29,Male,Allergic Rhinitis,Cetirizine,10,14,No,115/72,175
PAT-209,76,Male,Heart Disease,Clopidogrel,75,365,Yes,150/95,260
PAT-210,48,Female,Hypothyroidism,Levothyroxine,0.05,180,No,122/78,190
"""
df = pd.read_csv(StringIO(data))
df.to_csv("healthcare_patients_sample.csv", index=False)
df.head()

,PatientID,Age,Gender,Diagnosis,PrimaryMedication,DosageMg,TreatmentDurationDays,Readmitted,BloodPressure,CholesterolLevel
0,PAT-201,42,Male,Hypertension,Amlodipine,10.0,90,No,138/88,210
1,PAT-202,58,Female,Type 2 Diabetes,Metformin,500.0,180,No,128/80,195
2,PAT-203,67,Male,Heart Disease,Atorvastatin,40.0,120,Yes,145/92,245
3,PAT-204,35,Female,Asthma,Albuterol,2.0,30,No,118/75,180
4,PAT-205,71,Female,Osteoarthritis,Ibuprofen,400.0,60,Yes,132/84,205


In [3]:
dictionary = pd.DataFrame([
 ["PatientID","text","Pseudonymous patient code (PAT-xxx)","Identifier (pseudonymous)","Unique, format PAT-###","Not a real MRN"],
 ["Age","integer","Patient age in years","Demographic","0-120","Quasi-identifier"],
 ["Gender","text","Patient gender","Demographic","Male / Female","Quasi-identifier"],
 ["Diagnosis","text","Primary diagnosis","Clinical (sensitive)","Non-empty text",""],
 ["PrimaryMedication","text","Main medication prescribed","Clinical (sensitive)","Non-empty text",""],
 ["DosageMg","float","Dose in mg","Clinical","> 0, check per medication","Varies by drug (0.05 to 850)"],
 ["TreatmentDurationDays","integer","Treatment length in days","Clinical","1-365+",""],
 ["Readmitted","text","Hospital readmission flag","Clinical outcome","Yes / No",""],
 ["BloodPressure","text","Systolic/diastolic in mmHg","Clinical (sensitive)","Format ###/##","Will split into 2 columns"],
 ["CholesterolLevel","integer","Cholesterol in mg/dL","Clinical (sensitive)","100-400",""],
], columns=["field_name","data_type","description","category","expected_range","notes"])

dictionary.to_csv("data_dictionary.csv", index=False)
dictionary

,field_name,data_type,description,category,expected_range,notes
0,PatientID,text,Pseudonymous patient code (PAT-xxx),Identifier (pseudonymous),"Unique, format PAT-###",Not a real MRN
1,Age,integer,Patient age in years,Demographic,0-120,Quasi-identifier
2,Gender,text,Patient gender,Demographic,Male / Female,Quasi-identifier
3,Diagnosis,text,Primary diagnosis,Clinical (sensitive),Non-empty text,
4,PrimaryMedication,text,Main medication prescribed,Clinical (sensitive),Non-empty text,
5,DosageMg,float,Dose in mg,Clinical,"> 0, check per medication",Varies by drug (0.05 to 850)
6,TreatmentDurationDays,integer,Treatment length in days,Clinical,1-365+,
7,Readmitted,text,Hospital readmission flag,Clinical outcome,Yes / No,
8,BloodPressure,text,Systolic/diastolic in mmHg,Clinical (sensitive),Format ###/##,Will split into 2 columns
9,CholesterolLevel,integer,Cholesterol in mg/dL,Clinical (sensitive),100-400,


In [4]:
import re

exceptions = []   # issues inga collect aagum

# 1. Missing values
print("Missing values per column:")
print(df.isnull().sum())

# 2. Duplicate patient IDs and duplicate rows
print("\nDuplicate PatientIDs:", df["PatientID"].duplicated().sum())
print("Duplicate rows:", df.duplicated().sum())

# 3. Types
print("\nData types:")
print(df.dtypes)

# 4. Range checks
def log(pid, field, issue):
    exceptions.append({"PatientID": pid, "field": field, "issue": issue, "action": "flagged, not changed"})

for _, r in df.iterrows():
    pid = r["PatientID"]
    if not re.fullmatch(r"PAT-\d{3}", str(pid)):
        log(pid, "PatientID", "Format not PAT-###")
    if not 0 <= r["Age"] <= 120:
        log(pid, "Age", f"Out of range: {r['Age']}")
    if r["Gender"] not in ["Male", "Female"]:
        log(pid, "Gender", f"Unexpected value: {r['Gender']}")
    if r["DosageMg"] <= 0:
        log(pid, "DosageMg", f"Non-positive: {r['DosageMg']}")
    if r["TreatmentDurationDays"] <= 0:
        log(pid, "TreatmentDurationDays", f"Non-positive: {r['TreatmentDurationDays']}")
    if r["Readmitted"] not in ["Yes", "No"]:
        log(pid, "Readmitted", f"Unexpected value: {r['Readmitted']}")
    if not 100 <= r["CholesterolLevel"] <= 400:
        log(pid, "CholesterolLevel", f"Out of range: {r['CholesterolLevel']}")
    if not re.fullmatch(r"\d{2,3}/\d{2,3}", str(r["BloodPressure"])):
        log(pid, "BloodPressure", f"Bad format: {r['BloodPressure']}")

print("\nIssues found:", len(exceptions))
exception_log = pd.DataFrame(exceptions, columns=["PatientID","field","issue","action"])
exception_log.to_csv("exception_log.csv", index=False)
exception_log

Missing values per column:
PatientID                0
Age                      0
Gender                   0
Diagnosis                0
PrimaryMedication        0
DosageMg                 0
TreatmentDurationDays    0
Readmitted               0
BloodPressure            0
CholesterolLevel         0
dtype: int64

Duplicate PatientIDs: 0
Duplicate rows: 0

Data types:
PatientID                 object
Age                        int64
Gender                    object
Diagnosis                 object
PrimaryMedication         object
DosageMg                 float64
TreatmentDurationDays      int64
Readmitted                object
BloodPressure             object
CholesterolLevel           int64
dtype: object

Issues found: 0


,PatientID,field,issue,action


In [5]:
clean = df.copy()   # original df ah touch pannala

# Rule 1: text columns la extra spaces remove
for col in ["PatientID", "Gender", "Diagnosis", "PrimaryMedication", "Readmitted"]:
    clean[col] = clean[col].str.strip()

# Rule 2: BloodPressure ah systolic / diastolic nu 2 number columns ah pirikkalam
bp = clean["BloodPressure"].str.split("/", expand=True).astype(int)
clean["Systolic"] = bp[0]
clean["Diastolic"] = bp[1]

# Rule 3: BP sanity check (systolic > diastolic irukkanum)
bad_bp = clean[clean["Systolic"] <= clean["Diastolic"]]
print("Rows where systolic <= diastolic:", len(bad_bp))

# Rule 4: Readmitted Yes/No ah 1/0 flag ah maathalam (original column apdiye irukkum)
clean["Readmitted_Flag"] = clean["Readmitted"].map({"Yes": 1, "No": 0})

# Rule 5: Missing values fill pannala, clinical values invent pannala
print("Missing after cleaning:", clean.isnull().sum().sum())

clean.to_csv("clean_patient_records.csv", index=False)
clean

Rows where systolic <= diastolic: 0
Missing after cleaning: 0


,PatientID,Age,Gender,Diagnosis,PrimaryMedication,DosageMg,TreatmentDurationDays,Readmitted,BloodPressure,CholesterolLevel,Systolic,Diastolic,Readmitted_Flag
0,PAT-201,42,Male,Hypertension,Amlodipine,10.00,90,No,138/88,210,138,88,0
1,PAT-202,58,Female,Type 2 Diabetes,Metformin,500.00,180,No,128/80,195,128,80,0
2,PAT-203,67,Male,Heart Disease,Atorvastatin,40.00,120,Yes,145/92,245,145,92,1
3,PAT-204,35,Female,Asthma,Albuterol,2.00,30,No,118/75,180,118,75,0
4,PAT-205,71,Female,Osteoarthritis,Ibuprofen,400.00,60,Yes,132/84,205,132,84,1
5,PAT-206,50,Male,Type 2 Diabetes,Metformin,850.00,90,No,125/82,215,125,82,0
6,PAT-207,63,Female,Hypertension,Lisinopril,20.00,150,No,135/86,220,135,86,0
7,PAT-208,29,Male,Allergic Rhinitis,Cetirizine,10.00,14,No,115/72,175,115,72,0
8,PAT-209,76,Male,Heart Disease,Clopidogrel,75.00,365,Yes,150/95,260,150,95,1
9,PAT-210,48,Female,Hypothyroidism,Levothyroxine,0.05,180,No,122/78,190,122,78,0


In [6]:
from google.colab import files
for f in ["clean_patient_records.csv", "data_dictionary.csv", "exception_log.csv"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>